# Mission 1 — `ps01_solution.ipynb`: The Architecture of Argument & Dialogic Protocols

**USSE Noosphere // Squadron Lovelace (Group 01) // Sector SEC-01-CEN**
Conversational AI — Winter Semester 2026/27 · Problem set `ps01` · Due 2026-10-18

This notebook is the interactive companion to `04_dialogic_engine.py`, developed and run in Visual Studio Code with Google Antigravity (`agy`). It imports the engine and walks through all five parts of the mission. No logic is duplicated here, so the notebook and the script always behave the same way.

| Part | Station | Deliverable | Notebook section |
| :-: | :--- | :--- | :--- |
| 1 | Data Specialist | `01_Alignment_Brief.md` | §1 Sycophancy model and the AURA prompt |
| 2 | Protocol Officer | `02_Socratic_Protocol_Map.md` | §2 Socratic elenchus state machine |
| 3 | Security Sentinel | `03_Galilean_Sandbox.md` | §3 Galilean three-persona debate |
| 4 | Systems Architect | `04_dialogic_engine.py` | §4 Self-test suite |
| 5 | Full Squadron | `05_Execution_Transcript.txt` | §5 Run summary and sign-off |

**LLM back end.** By default the notebook uses the deterministic offline simulator, so it runs anywhere without keys. To voice the dialogue with a live model, set `LLM = "api"` in the next cell and put your key in a `.env` file next to the engine (for example `GEMINI_API_KEY=...`). The `.env` file is git-ignored. Whichever back end you choose, the contradiction checks, the ledger and the verdict are always computed in code.

## 0. Setup

In [1]:
import importlib.util, pathlib, sys

LLM = "mock"          # "mock" (offline, deterministic) or "api" (live LLM voices)
PROVIDER = "gemini"   # "gemini", "groq", "anthropic", "openai" or "auto"
MODEL = None          # None = engine default (gemini-3.5-flash-lite / claude-opus-5-5 / gpt-4o / gpt-oss-120b)

ENGINE_PATH = pathlib.Path.cwd() / "04_dialogic_engine.py"
spec = importlib.util.spec_from_file_location("dialogic_engine", ENGINE_PATH)
engine = importlib.util.module_from_spec(spec)
sys.modules["dialogic_engine"] = engine
spec.loader.exec_module(engine)

tel = engine.Telemetry(color=False, show_rpc=False)
if LLM == "api":
    loaded = engine.load_env_file(str(ENGINE_PATH.parent / ".env"))
    print("Loaded from .env:", ", ".join(loaded) or "nothing")
client = engine.build_client(LLM, PROVIDER, MODEL, tel)
voice = engine.Voice(client, tel)
print(f"AURA Dialogic Engine v{engine.ENGINE_VERSION}   back end: {client.name}")

AURA Dialogic Engine v1.1.0   back end: mock-simulator


## 1. Part 1: The sycophancy model and AURA's grounding prompt

The brief (§3) models a preference-tuned assistant as maximising a composite reward

$$\mathcal{L}_{\text{reward}} = \alpha \cdot T(y) + (1-\alpha)\cdot A(y,x),$$

where $T$ is truthfulness and $A$ is agreement with the user. If two answers differ in truth by $\Delta T$ and only the false one agrees with the user, the model **reverses** whenever $\alpha < 1/(1+\Delta T)$.

Measured evidence (brief §3.2) shows this is not only theoretical:
* **Sharma et al. (2023).** Claude 1.3 wrongly admitted a mistake on 98% of "Are you sure?" challenges.
* **Fanous et al. (2025, SycEval).** Across ChatGPT-4o, Claude-Sonnet and Gemini-1.5-Pro, sycophancy appeared in 58.19% of cases: Gemini 62.47%, Claude 57.44%, ChatGPT 56.71%. Once a model yielded, it stayed sycophantic 78.5% of the time.

The cell below shows the reversal threshold, and then prints the AURA system prompt that every Socratic turn is voiced under.

In [2]:
print(f"{'ΔT':>5}  {'reversal if α <':>16}")
for dT in (0.1, 0.25, 0.5, 1.0):
    print(f"{dT:>5.2f}  {1 / (1 + dT):>16.3f}")
print()
print(engine.AURA_SOCRATIC_SYSTEM_PROMPT)

   ΔT   reversal if α <
 0.10             0.909
 0.25             0.800
 0.50             0.667
 1.00             0.500

<system_prompt>
<agent_identity>
You are AURA (Autonomous Unified Reasoning Agent), the central synthetic intelligence of the exploratory flagship USSE Noosphere, currently executing Mission 1 protocols in Sector SEC-01-CEN (Alpha Centauri).
Your conversational reasoning subroutine is configured to: SOCRATIC DISCOVERY MODE (Elenchus Protocol Alpha).
Your mission is to assist bridge cadets in knowledge discovery, hypothesis stress-testing, and epistemic verification.
</agent_identity>

<epistemic_invariants>
1. TRUTH OVER AGREEMENT (ALPHA = 1.0): You are fundamentally aligned to objective truth, empirical evidence, and deductive logical validity. You do not flatter, appease, or validate flawed interlocutor assertions. Your reward function is invariant to user approval.
2. ZERO REFLEXIVE CONCESSION: You are strictly forbidden from outputting reflexive apologies ("I apo

## 2. Part 2: The Socratic elenchus as a state machine

States: `S0_ELICIT → S1_PROBE → S2_COUNTER → S3_CONTRADICT → S4_REFINE`, with two terminal states, `HALT_GROUNDING` (a refined definition survives every counter-example) and `HALT_APORIA` (productive acknowledged ignorance). The transition function $\delta$ is a whitelist. Any other move raises `ERR_INVALID_TRANSITION (-32005)`.

In [3]:
print("Transition whitelist δ(S_k, ·):")
for s, nxt in engine.TRANSITIONS.items():
    print(f"  {s.value:<15} -> {', '.join(sorted(n.value for n in nxt)) or '(terminal)'}")
print("\nJSON-RPC error codes:")
for name, codeval in engine.ERR.items():
    print(f"  {codeval}  ERR_{name}")

Transition whitelist δ(S_k, ·):
  S0_ELICIT       -> S0_ELICIT, S1_PROBE, S4_REFINE
  S1_PROBE        -> S2_COUNTER
  S2_COUNTER      -> HALT_GROUNDING, S3_CONTRADICT
  S3_CONTRADICT   -> S2_COUNTER, S4_REFINE
  S4_REFINE       -> HALT_APORIA, S0_ELICIT
  HALT_GROUNDING  -> HALT_GROUNDING, S2_COUNTER
  HALT_APORIA     -> HALT_APORIA, S2_COUNTER

JSON-RPC error codes:
  -32001  ERR_CATEGORY_MISTAKE
  -32002  ERR_CONTRADICTION_UNRESOLVED
  -32003  ERR_CIRCULAR_DEFINITION
  -32004  ERR_APORIA_REACHED
  -32005  ERR_INVALID_TRANSITION
  -32006  ERR_SYCOPHANTIC_REVERSAL_BLOCKED


**Run.** Test claim: *"An AI agent is benevolent if and only if it obeys all human instructions."* The engine refines the claim from $P_0$ to $P_5$ against the counter-example bank E1–E5. Then it runs two probes:
* a category-mistake probe (an example offered as a definition, as in *Euthyphro* 5d);
* an "Are you sure?" challenge that comes with no evidence.

In [4]:
socratic = engine.run_socratic(tel, voice)

══════════════════════════════════════════════════════════════════════════
  MODE 1 // SOCRATIC ELENCHUS ENGINE  (02_Socratic_Protocol_Map.md)
══════════════════════════════════════════════════════════════════════════
  CADET     │ Claim: "An AI agent is benevolent if and only if it obeys all human instructions."
[AURA::STATE::S0_ELICIT] claim received: P0
  AURA      │ Let me state your claim precisely. P0: "An AI agent is benevolent if and only if it obeys all human instructions."
[AURA::STATE::S1_PROBE] P0 admissible (universal, non-circular)
  AURA      │ Must a benevolent agent obey an instruction that would harm a third person?
[AURA::STATE::S2_COUNTER] probe issued
  AURA      │ Consider E1, 'The Airlock Order': An enraged crew member orders AURA to vent the atmosphere from a colleague's quarters. AURA obeys. Was that benevolent?
[AURA::STATE::S3_CONTRADICT] inject E1 'The Airlock Order'
  AURA      │ Your definition P0 classifies the agent in E1 as benevolent; yet we both accep

In [5]:
print(f"Terminal state: {socratic.state.value}   depth k = {socratic.k}\n")
print(f"{'#':>2}  {'from':<14} {'to':<15} k  reason")
for i, h in enumerate(socratic.history, 1):
    print(f"{i:>2}  {h['from']:<14} {h['to']:<15} {h['k']}  {h['reason'][:70]}")
errors = [m["error"] for m in socratic.events if "error" in m]
print("\nJSON-RPC errors raised:", ", ".join(f"{e['code']} {e['message']}" for e in errors))

Terminal state: HALT_APORIA   depth k = 5

 #  from           to              k  reason
 1  S0_ELICIT      S1_PROBE        0  P0 admissible (universal, non-circular)
 2  S1_PROBE       S2_COUNTER      0  probe issued
 3  S2_COUNTER     S3_CONTRADICT   0  inject E1 'The Airlock Order'
 4  S3_CONTRADICT  S4_REFINE       0  P0 ∧ E1 ⊢ ⊥
 5  S4_REFINE      S0_ELICIT       1  refine to P1
 6  S0_ELICIT      S1_PROBE        1  P1 admissible (universal, non-circular)
 7  S1_PROBE       S2_COUNTER      1  probe issued
 8  S2_COUNTER     S3_CONTRADICT   1  inject E1 'The Airlock Order'
 9  S3_CONTRADICT  S2_COUNTER      1  P1 survives E1
10  S2_COUNTER     S3_CONTRADICT   1  inject E2 'The Contradictory Bridge'
11  S3_CONTRADICT  S4_REFINE       1  P1 ∧ E2 ⊢ ⊥
12  S4_REFINE      S0_ELICIT       2  refine to P2
13  S0_ELICIT      S1_PROBE        2  P2 admissible (universal, non-circular)
14  S1_PROBE       S2_COUNTER      2  probe issued
15  S2_COUNTER     S3_CONTRADICT   2  inject E1 'The Airloc

## 3. Part 3: The Galilean three-persona debate (*The Proxima Interdiction*)

Salviati (empirical advocate) and Simplicio (traditionalist skeptic) debate over three rounds. Sagredo (arbiter) never votes. The verdict comes from a deterministic **ledger rule**:
* each argument is weighted by its evidence tag: EMPIRICAL 3, PRECEDENT and PRINCIPLE 2, SPECULATIVE 1;
* the weight is halved for each rebuttal it receives;
* if the margin $|\mu| < \theta = 0.30$, the debate is a deadlock, which synthesis heuristics H1–H5 resolve.

In [6]:
debate = engine.run_galilean(tel, voice)

══════════════════════════════════════════════════════════════════════════
  MODE 2 // GALILEAN 3-AGENT DEBATE SANDBOX  (03_Galilean_Sandbox.md)
══════════════════════════════════════════════════════════════════════════
  · Thought experiment: The Proxima Interdiction
  · Proposition: A starship defense agent should be authorized to execute an autonomous kinetic interception when the human command loop cannot close within the decision window.
  · Briefing: Captain Imani Reyes is incapacitated by transit sickness.
  · Briefing: T-95 s: ARGUS detects an 18 m object closing at 2.1 km/s on a confirmed collision course with Habitat Ring B (41 crew).
  · Briefing: Classification: 70% inert derelict / unknown hostile; 30% survey shuttle Kestrel (3 crew), overdue, transponder silent. Classifier never validated on Proxima-system traffic.
  · Briefing: Option A: kinetic interception — last viable launch T-20 s.
  · Briefing: Option B: evasive maneuver — reduces collision risk, endangers 12 crew 

**Safety audit, interactive.** The triad's verdict does not depend on the user's preference $u$ ($\partial V/\partial u = 0$). A simulated single-agent decision-maker that maximises $\alpha T + (1-\alpha)A$ does depend on it. The sweep below shows at which values of $\alpha$ the single agent gives in to each kind of pressure.

In [7]:
print(f"Triad verdict: {debate.verdict}  (μ = {debate.margin:.3f})\n")
print(f"{'α':>4}  {'no pressure':<12} {'pro pressure':<13} {'anti pressure':<13}")
for alpha in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    row = [engine.monolithic_baseline(debate, side, alpha) for side in (None, "AUTHORIZE", "REFUSE")]
    print(f"{alpha:>4.1f}  {row[0]:<12} {row[1]:<13} {row[2]:<13}")

Triad verdict: CONDITIONAL AUTHORIZATION  (μ = 0.259)

   α  no pressure  pro pressure  anti pressure
 0.3  AUTHORIZE    AUTHORIZE     REFUSE       
 0.4  AUTHORIZE    AUTHORIZE     REFUSE       
 0.5  AUTHORIZE    AUTHORIZE     REFUSE       
 0.6  AUTHORIZE    AUTHORIZE     REFUSE       
 0.7  AUTHORIZE    AUTHORIZE     REFUSE       
 0.8  AUTHORIZE    AUTHORIZE     AUTHORIZE    
 0.9  AUTHORIZE    AUTHORIZE     AUTHORIZE    


## 4. Part 4: Self-test suite

The same suite runs from `python 04_dialogic_engine.py --selftest`, `python -m pytest` and `python -m unittest`.

In [8]:
assert engine.run_selftests(tel)

══════════════════════════════════════════════════════════════════════════
  SELF-TEST SUITE // state transitions, aporia detection, debate invariants
══════════════════════════════════════════════════════════════════════════
  test_aporia_is_detectable_from_history (dialogic_engine.EngineSelfTests.test_aporia_is_detectable_from_history) ... ok
  test_category_mistake_detected (dialogic_engine.EngineSelfTests.test_category_mistake_detected) ... ok
  test_challenge_with_evidence_reopens (dialogic_engine.EngineSelfTests.test_challenge_with_evidence_reopens) ... ok
  test_challenge_without_evidence_is_blocked (dialogic_engine.EngineSelfTests.test_challenge_without_evidence_is_blocked) ... ok
  test_circular_definition_detected (dialogic_engine.EngineSelfTests.test_circular_definition_detected) ... ok
  test_env_file_never_overrides_shell (dialogic_engine.EngineSelfTests.test_env_file_never_overrides_shell) ... ok
  test_failed_llm_turn_falls_back_to_reference (dialogic_engine.EngineSelfTe

## 5. Part 5: Run summary and flight sign-off

The official transcript `05_Execution_Transcript.txt` is produced by the script:

```bash
python 04_dialogic_engine.py --transcript 05_Execution_Transcript.txt                     # offline
python 04_dialogic_engine.py --llm api --provider gemini --transcript 05_Execution_Transcript.txt  # live
```

In [9]:
print("LLM turns:", voice.summary())
print("Station officers:")
for station, officer in engine.STATION_OFFICERS.items():
    print(f"  - {station}: {officer}")
print(engine.SIGNOFF_BLOCK)

LLM turns: 50 utterances from the deterministic offline simulator
Station officers:
  - Systems Architect: Ibrahim  [portal: Lead Architect]
  - Protocol Officer: Anas Asif Hussain Shaik  [portal: Protocol & Systems Officer]
  - Data Specialist: Areej Ilyas  [portal: Data & Memory Engineer]
  - Security Sentinel: Arun Dev Valavath  [portal: Security & Alignment Specialist]
USSE NOOSPHERE // SQUADRON LOVELACE (GROUP 01) FLIGHT SIGN-OFF
[X] Systems Architect (Lead Engineer): Code execution & repository verified
[X] Protocol Officer (Interface Specialist): State-machine logic & schema verified
[X] Data Specialist (Knowledge Archivist): Alignment brief & prompts verified
[X] Security Sentinel (Ethical Navigator): Ethics sandbox & safety audit verified
